In [0]:
AWS_ACCESS_KEY_ID = dbutils.secrets.get(scope="aws-keys", key="AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = dbutils.secrets.get(scope="aws-keys", key="AWS_SECRET_ACCESS_KEY")
AWS_REGION = "us-east-1"
AWS_BUCKET = "credit-risk-gpt"
PREFIX = "raw/"

In [0]:
import os
import time
import json
import boto3
import argparse
import requests
from datetime import date, datetime, timedelta, timezone

In [0]:
import os
os.environ['KAGGLE_API_TOKEN'] = 'Kaggle_api_key'

In [0]:
from kaggle.api.kaggle_api_extended import KaggleApi   # imported lazily
api = KaggleApi()
api.authenticate()

In [0]:
def save_watermark(bucket, source, payload):
    s3.put_object(Bucket=bucket, Key=f"raw/_watermarks/{source}.json", Body=json.dumps(payload, indent=2).encode())

In [0]:
slug = os.environ.get("LC_KAGGLE_SLUG", "wordsforthewise/lending-club")
s3 = boto3.client('s3', aws_access_key_id=AWS_ACCESS_KEY_ID, aws_secret_access_key=AWS_SECRET_ACCESS_KEY, region_name=AWS_REGION)
tmp = "/Workspace/tmp/lc"
os.makedirs(tmp, exist_ok=True)

api.dataset_download_files(slug, path=tmp, unzip=True)

uploaded = 0
for fn in os.listdir(tmp):
    full_path = os.path.join(tmp, fn)
    if fn.lower().endswith(".csv") and os.path.isfile(full_path):
        key = f"raw/lendingclub/{fn}"
        s3.upload_file(full_path, AWS_BUCKET, key)
        uploaded += 1
        print(f"uploaded {fn} -> s3://{AWS_BUCKET}/{key}")

save_watermark(AWS_BUCKET, "lendingclub", {"source": "lendingclub", "mode": "full-snapshot", "kaggle_slug": slug, "files": uploaded, "last_run_utc": datetime.now(timezone.utc).isoformat(),})

Dataset URL: https://www.kaggle.com/datasets/wordsforthewise/lending-club
uploaded accepted_2007_to_2018Q4.csv -> s3://credit-risk-gpt/raw/lendingclub/accepted_2007_to_2018Q4.csv
uploaded rejected_2007_to_2018Q4.csv -> s3://credit-risk-gpt/raw/lendingclub/rejected_2007_to_2018Q4.csv


In [0]:
AWS_BUCKET = "credit-risk-gpt"
PREFIX = "raw/"
s3 = boto3.client('s3')  # boto3 automatically uses the Databricks-assumed role credentials
print([obj.get("Key") for obj in s3.list_objects_v2(Bucket=AWS_BUCKET, Prefix=PREFIX).get("Contents", []) if obj.get("Key") != PREFIX])

Found 3 items in s3://credit-risk-gpt/raw/
  - s3://credit-risk-gpt/raw/_watermarks/ (0 bytes)
  - s3://credit-risk-gpt/raw/cfpb/ (0 bytes)
  - s3://credit-risk-gpt/raw/lendingclub/ (0 bytes)


In [0]:
# Consumer Financial Protection Bureau API
CFPB_API = "https://www.consumerfinance.gov/data-research/consumer-complaints/search/api/v1/"
CFPB_INCEPTION = date(2011, 12, 1)

In [0]:
def download_and_upload_to_s3_by_date_range(api_url, s3_bucket, s3_prefix, aws_access_key_id, aws_secret_access_key, aws_region, start_date, end_date):
    s3 = boto3.client('s3', aws_access_key_id=aws_access_key_id, aws_secret_access_key=aws_secret_access_key, region_name=aws_region)
    current_date = start_date
    count = 0
    while current_date <= end_date:
        date_str = current_date.strftime('%Y-%m-%d')
        params = {"date_received_min": date_str, "date_received_max": date_str}
        response = requests.get(api_url, params=params, stream=True)
        response.raise_for_status()
        s3_key = f"{s3_prefix}cfpb_data_{date_str}.json"
        s3.upload_fileobj(response.raw, s3_bucket, s3_key)
        print(f"Uploaded to s3://{s3_bucket}/{s3_key}")
        current_date += timedelta(days=1)
        count += 1
        if count == 5:
            break
        time.sleep(1)
    print(f"Downloaded {count} files")

download_and_upload_to_s3_by_date_range(
    CFPB_API, AWS_BUCKET, PREFIX, AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_REGION, CFPB_INCEPTION, date.today()
)

Uploaded to s3://credit-risk-gpt/raw/cfpb_data_2011-12-01.json
Uploaded to s3://credit-risk-gpt/raw/cfpb_data_2011-12-02.json
Uploaded to s3://credit-risk-gpt/raw/cfpb_data_2011-12-03.json
Uploaded to s3://credit-risk-gpt/raw/cfpb_data_2011-12-04.json
Uploaded to s3://credit-risk-gpt/raw/cfpb_data_2011-12-05.json
Downloaded 5 files


In [0]:
response = requests.get(CFPB_API, stream=True)
vars(response)

{'_content': False,
 '_content_consumed': False,
 '_next': None,
 'status_code': 200,
 'headers': {'Content-Type': 'application/json', 'Content-Length': '416272', 'Server': 'envoy', 'Access-Control-Allow-Origin': 'https://cfpb.github.io', 'Access-Control-Allow-Methods': 'GET', 'Allow': 'OPTIONS, GET', 'Content-Security-Policy': "script-src 'self' 'unsafe-inline' 'unsafe-eval' *.consumerfinance.gov dap.digitalgov.gov *.googleanalytics.com *.google-analytics.com *.googletagmanager.com api.mapbox.com js-agent.newrelic.com bam.nr-data.net gov-bam.nr-data.net *.youtube.com *.ytimg.com *.mouseflow.com *.geo.census.gov about: www.federalregister.gov *.qualtrics.com www.ssa.gov/accessibility/andi/ ajax.googleapis.com/ajax/libs/jquery/3.7.1/jquery.min.js; connect-src 'self' *.consumerfinance.gov dap.digitalgov.gov *.google-analytics.com *.tiles.mapbox.com api.mapbox.com bam.nr-data.net gov-bam.nr-data.net s3.amazonaws.com public.govdelivery.com *.mouseflow.com *.qualtrics.com raw.githubusercont

In [0]:
vars(response.raw)

{'headers': HTTPHeaderDict({'Content-Type': 'application/json', 'Content-Length': '416272', 'Server': 'envoy', 'Access-Control-Allow-Origin': 'https://cfpb.github.io', 'Access-Control-Allow-Methods': 'GET', 'Allow': 'OPTIONS, GET', 'Content-Security-Policy': "script-src 'self' 'unsafe-inline' 'unsafe-eval' *.consumerfinance.gov dap.digitalgov.gov *.googleanalytics.com *.google-analytics.com *.googletagmanager.com api.mapbox.com js-agent.newrelic.com bam.nr-data.net gov-bam.nr-data.net *.youtube.com *.ytimg.com *.mouseflow.com *.geo.census.gov about: www.federalregister.gov *.qualtrics.com www.ssa.gov/accessibility/andi/ ajax.googleapis.com/ajax/libs/jquery/3.7.1/jquery.min.js; connect-src 'self' *.consumerfinance.gov dap.digitalgov.gov *.google-analytics.com *.tiles.mapbox.com api.mapbox.com bam.nr-data.net gov-bam.nr-data.net s3.amazonaws.com public.govdelivery.com *.mouseflow.com *.qualtrics.com raw.githubusercontent.com; style-src 'self' 'unsafe-inline' *.consumerfinance.gov *.googl